In [44]:
import pandas as pd
from pybaseball import playerid_reverse_lookup, retrosheet, team_game_logs, schedule_and_record

# help(retrosheet)

In [18]:
season_data = pd.read_csv("data/statcast_2024.csv", parse_dates=["game_date"])

In [33]:
print(season_data["inning_topbot"].tail())

708127    Top
708128    Top
708129    Top
708130    Top
708131    Top
Name: inning_topbot, dtype: object


In [14]:
columns = season_data.columns
columns = sorted(season_data.columns)

for column in season_data.columns:
    print(column)

pitch_type
game_date
release_speed
release_pos_x
release_pos_z
player_name
batter
pitcher
events
description
spin_dir
spin_rate_deprecated
break_angle_deprecated
break_length_deprecated
zone
des
game_type
stand
p_throws
home_team
away_team
type
hit_location
bb_type
balls
strikes
game_year
pfx_x
pfx_z
plate_x
plate_z
on_3b
on_2b
on_1b
outs_when_up
inning
inning_topbot
hc_x
hc_y
tfs_deprecated
tfs_zulu_deprecated
umpire
sv_id
vx0
vy0
vz0
ax
ay
az
sz_top
sz_bot
hit_distance_sc
launch_speed
launch_angle
effective_speed
release_spin_rate
release_extension
game_pk
fielder_2
fielder_3
fielder_4
fielder_5
fielder_6
fielder_7
fielder_8
fielder_9
release_pos_y
estimated_ba_using_speedangle
estimated_woba_using_speedangle
woba_value
woba_denom
babip_value
iso_value
launch_speed_angle
at_bat_number
pitch_number
pitch_name
home_score
away_score
bat_score
fld_score
post_away_score
post_home_score
post_bat_score
post_fld_score
if_fielding_alignment
of_fielding_alignment
spin_axis
delta_home_win_exp
d

In [5]:
print(season_data["game_pk"].head())

0    775296
1    775296
2    775296
3    775296
4    775296
Name: game_pk, dtype: int64


In [25]:
df = playerid_reverse_lookup([502110], key_type='mlbam')
name = df['name_first'].iloc[0] + " " + df['name_last'].iloc[0]


In [26]:
print(name)

j. d. martinez


In [31]:
single_player = season_data[season_data["pitcher"] == 502171]
print(single_player.head())

     pitch_type  game_date  release_speed  release_pos_x  release_pos_z  \
4408         FS 2024-10-14           89.5          -2.16           5.59   
4409         SI 2024-10-14           94.1          -2.17           5.56   
4410         KC 2024-10-14           82.4          -2.14           5.78   
4411         SI 2024-10-14           93.3          -2.31           5.52   
4412         SI 2024-10-14           95.4          -2.09           5.54   

     player_name  batter  pitcher events    description  ...  \
4408  Cobb, Alex  683011   502171   walk           ball  ...   
4409  Cobb, Alex  683011   502171    NaN  called_strike  ...   
4410  Cobb, Alex  683011   502171    NaN           ball  ...   
4411  Cobb, Alex  683011   502171    NaN           ball  ...   
4412  Cobb, Alex  683011   502171    NaN           ball  ...   

      batter_days_until_next_game  api_break_z_with_gravity  api_break_x_arm  \
4408                          1.0                      2.65             1.58   
4409

In [19]:
game_dates = season_data["game_date"]
print(game_dates.head())

0   2024-10-30
1   2024-10-30
2   2024-10-30
3   2024-10-30
4   2024-10-30
Name: game_date, dtype: datetime64[ns]


In [22]:
game_dates = sorted(game_dates)
print(game_dates[:5])       

[Timestamp('2024-04-01 00:00:00'), Timestamp('2024-04-01 00:00:00'), Timestamp('2024-04-01 00:00:00'), Timestamp('2024-04-01 00:00:00'), Timestamp('2024-04-01 00:00:00')]


In [29]:
batter_id = 502110  # JD Martinez, replace with your batter id

sub = season_data[season_data["batter"] == batter_id].copy()

# Prefer rows with non-null events within each PA
sub["events_notnull"] = sub["events"].notna()
sub = sub.sort_values(
    ["game_pk", "at_bat_number", "events_notnull"],
    ascending=[True, True, False]
)

# One row per PA per player
pa_rows = sub.groupby(["game_pk", "at_bat_number", "batter"], as_index=False).first()

out = pa_rows[["game_date","batter", "pitcher", "events", "description"]].copy()

out

,game_date,batter,pitcher,events,description
0,2024-07-03,502110,680730,field_out,hit_into_play
1,2024-07-03,502110,680730,double,hit_into_play
2,2024-07-03,502110,680730,field_out,hit_into_play
3,2024-07-03,502110,640451,strikeout,called_strike
4,2024-07-01,502110,669022,walk,ball
...,...,...,...,...,...
514,2024-10-02,502110,593423,field_out,hit_into_play
515,2024-10-02,502110,606303,walk,ball
516,2024-10-02,502110,605452,field_out,hit_into_play
517,2024-10-01,502110,676879,single,hit_into_play


In [45]:
sched = schedule_and_record(2024, "STL")
sched.head()

http://www.baseball-reference.com/teams/STL/2024-schedule-scores.shtml


/Users/wfurlow/miniforge3/envs/baseball/lib/python3.12/site-packages/pybaseball/team_results.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Attendance'].replace(r'^Unknown$', np.nan, regex=True, inplace = True) # make this a NaN so the column can benumeric


,Date,Tm,Home_Away,Opp,W/L,R,RA,Inn,W-L,Rank,GB,Win,Loss,Save,Time,D/N,Attendance,cLI,Streak,Orig. Scheduled
1,"Thursday, Mar 28",STL,@,LAD,L,1.0,7.0,9.0,0-1,4.0,1.0,Glasnow,Mikolas,Yarbrough,2:23,D,52667.0,.94,-1,None
2,"Friday, Mar 29",STL,@,LAD,L,3.0,6.0,9.0,0-2,5.0,2.0,Miller,Thompson,Phillips,2:23,N,47524.0,.90,-2,None
3,"Saturday, Mar 30",STL,@,LAD,W,6.0,5.0,10.0,1-2,4.0,2.0,Helsley,Hurt,Gallegos,3:17,N,45019.0,.85,1,None
4,"Sunday, Mar 31",STL,@,LAD,L,4.0,5.0,9.0,1-3,5.0,3.0,Crismatt,King,Hudson,2:41,D,41014.0,.90,-1,None
5,"Monday, Apr 1",STL,@,SDP,W,6.0,2.0,9.0,2-3,5.0,3.0,Gibson,Waldron,None,2:45,N,37566.0,.86,1,None


In [46]:
sched.columns

Index(['Date', 'Tm', 'Home_Away', 'Opp', 'W/L', 'R', 'RA', 'Inn', 'W-L',
       'Rank', 'GB', 'Win', 'Loss', 'Save', 'Time', 'D/N', 'Attendance', 'cLI',
       'Streak', 'Orig. Scheduled'],
      dtype='object')

In [47]:
batting_logs = team_game_logs(2019, "ATL")


RuntimeError: Table with expected id not found on scraped page.